# Otto — Simulação, Modelagem e Validação com ROS 2, Gazebo, RViz, MATLAB e/ou Octave

**Repositório (código-fonte):** https://github.com/InovaMech/otto

**Imagem Docker (ambiente pronto):** https://hub.docker.com/r/inovamech/otto-sim

Este material apresenta o fluxo completo do projeto Otto, desde a preparação do computador (WSL2, virtualização, Docker, VSCode) até a modelagem CAD, simulação e controle em ROS 2 e Gazebo.

O objetivo é fornecer um ambiente reproduzível para ensino e validação de robótica bípede, mesmo para quem nunca usou Docker, WSL ou ROS 2 antes.

> **Antes de começar:** leia a seção 1 com atenção. Ela existe porque a maioria dos erros de "não funciona" em sala de aula vem da ordem errada de instalação (ex.: instalar Docker antes do WSL2, ou esquecer de habilitar a virtualização na BIOS) — não do projeto Otto em si.


## 1. Como o ambiente é organizado (leia antes de instalar)

Antes de instalar qualquer coisa, é importante entender **onde vive cada parte** do projeto. Essa separação é intencional e evita confusão mais tarde:

| Onde | O que tem | Por quê |
|---|---|---|
| **Imagem Docker** `inovamech/otto-sim` | ROS 2 Jazzy, Gazebo Harmonic, RViz, colcon, todas as bibliotecas e dependências já compiladas | Ambiente pesado e estável — você baixa uma vez e não precisa reinstalar nada manualmente |
| **Repositório GitHub** `InovaMech/otto` | Apenas a pasta `src/` — o código-fonte dos pacotes ROS 2 do Otto (URDF/Xacro, launch files, scripts) | Código leve, versionado, que muda com frequência conforme a aula evolui |

Dentro do container, o workspace colcon mora em `/ros2_ws/`. A pasta `/ros2_ws/src` é a **única parte substituída** pelo código do seu computador (host) através de um *bind mount* — é assim que você consegue editar arquivos no VSCode (no Windows/WSL) e ver o efeito imediatamente dentro do container, sem precisar reconstruir a imagem.

```
┌────────────────────────────┐        ┌────────────────────────────────┐
│   SEU COMPUTADOR (host)     │        │   CONTAINER (imagem otto-sim)   │
│                              │        │                                  │
│  ~/otto/                    │        │  /ros2_ws/                      │
│   └── src/  ───────────────────mount────▶  └── src/   (substituída)     │
│        (clonado do GitHub)  │        │      build/   (já vem na imagem)│
│                              │        │      install/ (já vem na imagem)│
└────────────────────────────┘        └────────────────────────────────┘
```

>  **Ponto crítico que causa erro de "pacote não encontrado":** a imagem já vem com `build/` e `install/` pré-compilados — mas eles foram gerados a partir do `src/` *de fábrica* da imagem. Quando você monta seu `src/` do GitHub por cima, o ROS 2 ainda está "lembrando" da compilação antiga. **Por isso, depois de montar o volume pela primeira vez (ou sempre que atualizar o `src/` com `git pull`), é obrigatório apagar `build/`, `install/`, `log/` e recompilar com `colcon build`.** Isso está destacado no passo 4.3 — não pule essa etapa.

Essa arquitetura mantém o repositório GitHub **limpo** (só código-fonte, sem arquivos binários de build) e a imagem Docker **completa** (sem precisar baixar dependências a cada aula). O preço dessa organização é só esse um passo extra de recompilar — e é isso que o tutorial deixa explícito a partir de agora.


## 1.1. Requisitos do computador (antes de instalar)

- Windows 10 64-bit (build 19045 / 22H2) ou Windows 11 64-bit (build 22631 / 23H2 ou superior). Qualquer edição (Home, Pro, Enterprise, Education) funciona com o backend WSL 2.
- Processador 64-bit com suporte a virtualização (Intel VT-x ou AMD-V) — praticamente todo processador de 2015 em diante tem isso, mas costuma vir **desabilitado na BIOS/UEFI** de fábrica.
- Pelo menos 8 GB de RAM (mínimo absoluto 4 GB, mas Gazebo + RViz + ROS 2 pedem mais).
- Espaço em disco livre: ~10 GB (imagem Docker + expansão + cache).
- Conexão à internet estável para o primeiro `docker pull` (download de ~1,6 GB).

A ordem correta de instalação é:

1. **Habilitar a virtualização na BIOS/UEFI** (passo 1.2)
2. **Instalar o WSL2** (passo 1.3)
3. **Instalar o Docker Desktop** configurado para usar o backend WSL2 (passo 1.4)
4. **Instalar o VSCode + extensões** (passo 1.5)
5. Só então clonar o repositório e rodar o container (seções 2 em diante)

Pular a ordem (por exemplo, instalar o Docker Desktop antes de habilitar a virtualização) é a causa mais comum do erro *"Docker Desktop - WSL kernel error"* ou *"Hardware assisted virtualization and data execution protection must be enabled in the BIOS"*.


## 1.2. Habilitar a virtualização (BIOS/UEFI)

Esse passo evita a falha mais comum ao abrir o Docker Desktop pela primeira vez.

### Verificar se já está habilitada (sem precisar entrar na BIOS)

Abra o **Gerenciador de Tarefas** (`Ctrl + Shift + Esc`) → aba **Desempenho** → **CPU**. No canto inferior direito deve aparecer:

```
Virtualização: Habilitado
```

Se aparecer **Desabilitado**, siga os passos abaixo.

### Como habilitar

1. Reinicie o computador e entre na BIOS/UEFI (geralmente pressionando `Del`, `F2`, `F10` ou `Esc` repetidamente durante a inicialização — varia por fabricante).
2. Procure por uma opção chamada **Intel VT-x**, **Intel Virtualization Technology**, **AMD-V** ou **SVM Mode** (em placas AMD). Geralmente fica em **Advanced → CPU Configuration** ou **Security**.
3. Habilite a opção, salve (`F10` na maioria das BIOS) e reinicie.

### Habilitar as features do Windows necessárias

Abra o **PowerShell como Administrador** e rode:

```powershell
dism.exe /online /enable-feature /featurename:Microsoft-Windows-Subsystem-Linux /all /norestart
dism.exe /online /enable-feature /featurename:VirtualMachinePlatform /all /norestart
```

Reinicie o computador após esse comando.

Para confirmar que as features estão ativas:

```powershell
Get-WindowsOptionalFeature -Online | Where-Object {$_.FeatureName -like "*Linux*" -or $_.FeatureName -like "*Virtual*"} | Select-Object FeatureName, State
```

Ambas devem aparecer como `Enabled`.


## 1.3. Instalar o WSL2

Com a virtualização já habilitada, abra o **PowerShell como Administrador** e rode:

```powershell
wsl --install
```

Esse comando único instala o WSL2, baixa o kernel Linux da Microsoft e instala a distribuição Ubuntu (padrão). Reinicie o computador quando solicitado.

> Se o WSL já estiver instalado (de algum uso anterior), rode `wsl --update` em vez de `wsl --install`, e confirme a versão com `wsl --version` (recomendado: 2.1.5 ou superior).

Depois de reiniciar, abra o **Ubuntu** pelo menu Iniciar uma vez para finalizar a configuração e criar seu usuário/senha Linux.

Para confirmar que está tudo certo:

```powershell
wsl --status
wsl -l -v
```

A distribuição deve aparecer com **Version 2** (não Version 1).

>  **Nota:** você não precisa instalar nada de ROS 2 ou Gazebo dentro dessa distribuição Ubuntu do WSL2 manualmente — tudo isso já vem dentro da imagem Docker `inovamech/otto-sim`. O WSL2 aqui serve apenas como a camada de Linux que o Docker Desktop usa por baixo dos panos.


## 1.4. Instalar o Docker Desktop (backend WSL2)

1. Baixe o instalador em: https://www.docker.com/products/docker-desktop
2. Execute o `Docker Desktop Installer.exe`.
3. Na tela de configuração, deixe marcada a opção **"Use WSL 2 instead of Hyper-V"** (geralmente já vem marcada por padrão em sistemas que suportam WSL2 — essa é a opção recomendada para este projeto).
4. Conclua a instalação e reinicie se solicitado.
5. Abra o Docker Desktop. Vá em **Settings → General** e confirme que **"Use the WSL 2 based engine"** está marcado.
6. Vá em **Settings → Resources → WSL Integration** e habilite a integração com a sua distribuição (Ubuntu).

### Verificar a instalação

Abra um terminal Ubuntu (WSL) ou PowerShell e rode:

```bash
docker --version
docker run hello-world
```

Se aparecer a mensagem de boas-vindas do `hello-world`, o Docker está funcionando corretamente.

> ⚠️ **Erro comum:** se aparecer *"Hardware assisted virtualization and data execution protection must be enabled in the BIOS"*, volte ao passo 0.2 — a virtualização não foi habilitada corretamente na BIOS, mesmo que as features do Windows estejam ativas.


## 1.5. Instalar o VSCode e as extensões necessárias

1. Baixe e instale o VSCode: https://code.visualstudio.com/
2. Abra o VSCode e instale as seguintes extensões (aba **Extensions**, `Ctrl+Shift+X`, ou pelos links abaixo):

| Extensão | Identificador | Para quê serve |
|---|---|---|
| **WSL** | `ms-vscode-remote.remote-wsl` | Abrir o VSCode direto dentro do WSL2, editando os arquivos como se estivessem no Linux |
| **Dev Containers** | `ms-vscode-remote.remote-containers` | Abrir o VSCode *dentro* do container Docker, com IntelliSense, terminal e debug já configurados para o ambiente ROS 2 |
| **Docker** | `ms-azuretools.vscode-docker` | Gerenciar imagens, containers e volumes pela interface do VSCode |
| **ROS** | `ms-iot.vscode-ros` | Syntax highlighting, autocomplete e debug para arquivos `.launch.py`, `.urdf`, `.xacro` e nós ROS 2 |
| **Python** | `ms-python.python` | Suporte a Python (usado pelos scripts como `walk.py` e pelos notebooks) |
| **XML** | `redhat.vscode-xml` | Syntax highlighting e validação para arquivos `.urdf`/`.xacro` (que são XML) |

Você pode instalar todas de uma vez pelo terminal (PowerShell ou WSL, com o VSCode já no PATH):

```bash
code --install-extension ms-vscode-remote.remote-wsl
code --install-extension ms-vscode-remote.remote-containers
code --install-extension ms-azuretools.vscode-docker
code --install-extension ms-iot.vscode-ros
code --install-extension ms-python.python
code --install-extension redhat.vscode-xml
```

### Recomendação de fluxo de trabalho

- Edite os arquivos do projeto (pasta `src/`) **a partir do WSL2**, usando `code .` dentro do terminal Ubuntu — isso abre o VSCode já conectado ao sistema de arquivos Linux, evitando problemas de performance e de permissões que ocorrem ao editar arquivos Linux a partir do Windows diretamente.
- Para editar arquivos *dentro* do container em execução (por exemplo, depurar algo que só existe ali), use a extensão **Dev Containers** → **Attach to Running Container**.


## 2. Visão Geral

O Otto é um robô bípede educacional simulado.

O fluxo do projeto é:

- Fusion 360 → modelagem
- Xacro/URDF → descrição do robô
- ROS 2 → controle
- Gazebo → simulação física
- RViz → visualização
- MATLAB/Octave → análise


## 3. Ambiente de Execução (Docker)

O ambiente já está pronto no Docker Hub — você não precisa instalar ROS 2, Gazebo ou nenhuma dependência manualmente.

### Imagem

```bash
inovamech/otto-sim:latest
```

Repositório da imagem: https://hub.docker.com/r/inovamech/otto-sim

### Tamanho

- Download: ~1,6 GB
- Expansão interna: ~7 GB

Inclui ROS 2 Jazzy + Gazebo Harmonic + RViz + colcon + ros2_control + dependências.

> Lembrete: o **código-fonte** do robô (pacotes em `src/`) **não vem dentro da imagem para edição contínua** — ele vive no repositório GitHub e é montado como volume (seção 0 explica o porquê). A imagem fornece o *ambiente*; o GitHub fornece o *código*.


## 4. Clonar o repositório e baixar a imagem

### 4.1. Clonar o repositório (dentro do WSL2)

Abra um terminal Ubuntu (WSL2) — pelo menu Iniciar ou digitando `wsl` no PowerShell — e rode:

```bash
cd ~
git clone https://github.com/InovaMech/otto.git
```

Isso cria a pasta `~/otto/` contendo apenas a pasta `src/` (o código-fonte dos pacotes ROS 2 do Otto). É justamente essa pasta `~/otto/src` que será montada dentro do container no passo 3.3.

### 4.2. Baixar a imagem Docker

```bash
docker pull inovamech/otto-sim:latest
```

### 4.3. Rodar o container

```bash
docker run -it \
  --name otto \
  --net=host \
  --privileged \
  -e DISPLAY=$DISPLAY \
  -v /tmp/.X11-unix:/tmp/.X11-unix \
  -v /home/inovamech/otto/src:/ros2_ws/src \
  inovamech/otto-sim:latest
```

Note que o volume aponta para `~/otto/src` (a pasta `src/` dentro do repositório clonado), não para `~/otto` diretamente — é exatamente o que substitui `/ros2_ws/src` dentro do container.

### 4.4. Entrar no container (em outro terminal, quando precisar)

```bash
docker start otto && docker exec -it otto bash
```


## 5. Workspace

### 5.1. O que já vem pronto

A imagem já contém o workspace `/ros2_ws/` compilado de fábrica (com o `src/` original da imagem).

### 5.2. Recompilação obrigatória após montar o volume

Como visto na seção 0, ao montar `~/otto/src:/ros2_ws/src`, o conteúdo de `src/` dentro do container passa a ser o código do GitHub — mas `build/` e `install/` ainda refletem a compilação antiga. **Sempre que você montar o volume pela primeira vez, ou depois de um `git pull` que traga pacotes novos/renomeados, rode estes comandos dentro do container:**

```bash
cd /ros2_ws
rm -rf build install log
colcon build --symlink-install
source /opt/ros/jazzy/setup.bash
source install/setup.bash
```

Sem esse passo, comandos como `ros2 launch otto_description gazebo.launch.py` podem falhar com `Package 'otto_description' not found` (se o pacote mudou de nome) ou, pior, rodar uma versão **desatualizada** do código sem avisar.

### 5.3. Em compilações futuras (já recompilado uma vez)

Se você não alterou nada em `src/` desde a última compilação dentro deste mesmo container, basta:

```bash
source /opt/ros/jazzy/setup.bash
source /ros2_ws/install/setup.bash
```

> **Dica:** como `build/`, `install/` e `log/` vivem *dentro do container* (não no host), eles são perdidos se você remover o container (`docker rm otto`). Isso é esperado — ao criar o container de novo, repita o passo 4.2 uma vez.


## 6. Executar simulação no Gazebo

```bash
ros2 launch otto_description gazebo.launch.py
```

Este launch executa o pipeline completo de simulação:

* inicia o Gazebo Harmonic
* realiza o spawn do robô Otto no ambiente
* inicializa o ros2_control (controller_manager e hardware interface)
* ativa os controladores responsáveis pelas juntas
* publica automaticamente `/joint_states` via `joint_state_broadcaster`
* configura a comunicação ROS 2 ↔ Gazebo (`ros_gz_bridge`)

O Gazebo atua como ambiente físico principal, sendo a fonte de estados dinâmicos do robô durante a simulação.

## 7. Visualização no RViz

O RViz pode ser executado de forma independente do Gazebo, caso necessário para depuração ou inspeção do modelo.

```bash
ros2 launch otto_description display.launch.py
```

Este modo é utilizado para:

* validar a estrutura URDF/Xacro
* verificar a árvore TF
* inspecionar o modelo do robô sem simulação física
* depurar posições de juntas com `joint_state_publisher_gui`

Quando o Gazebo estiver em execução, o RViz pode ser usado simultaneamente para visualização do estado do robô em tempo real.


## 8. Executar caminhada (walk.py)

Em outro terminal, primeiro entre no container (`docker exec -it otto bash`), depois:

```bash
source /opt/ros/jazzy/setup.bash && \
source /ros2_ws/install/setup.bash && \
python3 /ros2_ws/src/otto_description/scripts/walk.py
```

Esse nó:
- gera movimento senoidal
- simula marcha bípede


## 9. Ordem correta de execução (resumo)

1. Habilitar virtualização + instalar WSL2 + Docker Desktop + VSCode (seção 0 — feito uma única vez por computador)
2. Clonar o repositório `~/otto` (seção 3.1 — feito uma vez, ou `git pull` para atualizar)
3. Subir o container com o volume montado (seção 3.3)
4. Recompilar o workspace dentro do container (seção 4.2 — sempre que o container for novo ou o `src/` mudar)
5. Iniciar Gazebo (seção 5)
6. Rodar `walk.py` (seção 7)
7. (opcional) abrir RViz (seção 6)

Fluxo mínimo:

```
Clonar repo → docker run (com volume) → colcon build → Gazebo → controle → walk → movimento
```


## 10. MATLAB / Octave

### Animação da caminhada do Otto

Este script anima um ciclo de caminhada do robô Otto a partir do seu modelo URDF,
carregado no MATLAB com a ferramenta `importrobot`. A ideia central é a mesma usada
em qualquer animação 3D: definir um pequeno conjunto de **posturas-chave** (keyframes)
— como "pé esquerdo levanta", "perna recua", "pé abaixa" — e depois gerar
automaticamente os quadros intermediários entre cada par de posturas por meio de
**interpolação linear**, criando a ilusão de movimento suave em vez de saltos abruptos.
O robô é redesenhado a cada quadro com a função `show`, e o comando `drawnow` força a
atualização imediata da tela, dando o efeito de animação em tempo real. O ciclo completo
de caminhada é repetido várias vezes, alternando entre o lado esquerdo e o direito do robô.

```C
% ===== CARREGAMENTO DO ROBÔ =====
% Lê o arquivo URDF e monta a árvore cinemática (links + juntas) do Otto
robot = importrobot('Z:\home\inovamech\otto\ros2_ws\src\otto_description\urdf\otto.urdf');

% Pega a configuração "neutra" do robô (todas as juntas em 0 rad)
config = homeConfiguration(robot);

% ===== CONFIGURAÇÃO DA JANELA 3D =====
figure;
show(robot, config, 'Frames', 'off', 'PreservePlot', false); % desenha o robô na pose atual
% 'Frames','off'      -> esconde os eixinhos de referência de cada junta
% 'PreservePlot',false -> não acumula desenhos antigos, redesenha do zero a cada chamada

xlim([-0.10 0.10]); ylim([-0.10 0.10]); zlim([-0.05 0.15]); % limites dos eixos (em metros)
view(45, 20); % ângulo da câmera: 45° de azimute, 20° de elevação

% ===== TABELA DE POSTURAS-CHAVE (KEYFRAMES) =====
% Cada linha = uma postura do ciclo de caminhada
% Cada coluna = o ângulo (em radianos) de uma junta específica
% (confirmar a ordem real das juntas no URDF: aqui assume-se
%  coluna 1 = quadril esq., 2 = joelho esq., 3 = quadril dir., 4 = joelho dir.)
posturas = [
    0.0   0.0   0.0   0.0;   % 1 - Neutro
    0.0   0.2   0.0   0.0;   % 2 - Pé esquerdo levanta
    -0.3   0.2   0.0   0.0;  % 3 - Perna esquerda recua
    -0.3   0.0   0.0   0.0;  % 4 - Pé esquerdo abaixa
    0.0   0.0   0.0   0.0;   % 5 - Perna esquerda volta ao neutro
    0.0   0.0   0.0   0.2;   % 6 - Pé direito levanta
    0.0   0.0  -0.3   0.2;   % 7 - Perna direita recua
    0.0   0.0  -0.3   0.0;   % 8 - Pé direito abaixa
    0.0   0.0   0.0   0.0;   % 9 - Perna direita volta ao neutro
];

% Rótulos de cada postura, usados no título do gráfico (mesma ordem da tabela acima)
titulos = {
    'Neutro'
    'Pe esquerdo levanta'
    'Perna esquerda recua'
    'Pe esquerdo abaixa'
    'Perna esquerda volta neutro'
    'Pe direito levanta'
    'Perna direita recua'
    'Pe direito abaixa'
    'Perna direita volta neutro'
};

n_cycles = 5; % quantas vezes o ciclo completo de caminhada será repetido

% ===== LOOP PRINCIPAL DA ANIMAÇÃO =====
for c = 1:n_cycles                     % repete o ciclo de caminhada inteiro
    for i = 1:size(posturas, 1)        % percorre cada postura-chave da tabela

        % Define de onde (p_prev) e para onde (p_next) o robô vai se mover
        if i == 1
            p_prev = posturas(end,:);  % se for a 1ª postura, "vem" da última (fecha o ciclo sem salto)
        else
            p_prev = posturas(i-1,:);
        end
        p_next = posturas(i,:);

        % Gera quadros intermediários entre p_prev e p_next (movimento suave)
        steps = 8;
        for s = 1:steps
            t = s/steps;                        % fração do caminho percorrido (de 1/8 até 1)
            p = p_prev + t*(p_next - p_prev);    % interpolação linear (LERP) entre as duas posturas

            % Aplica o ângulo interpolado a cada uma das 4 juntas
            config(1).JointPosition = p(1);
            config(2).JointPosition = p(2);
            config(3).JointPosition = p(3);
            config(4).JointPosition = p(4);

            % Redesenha o robô na nova postura
            show(robot, config, 'Frames', 'off', 'PreservePlot', false);
            xlim([-0.15 0.15]); ylim([-0.15 0.15]); zlim([-0.05 0.20]); % reforça os limites (show pode resetar)
            view(45, 20); % reforça o ângulo de câmera

            % Título dinâmico: mostra ciclo atual e fase do passo
            title(sprintf('Ciclo %d/%d - %s', c, n_cycles, titulos{i}));

            drawnow; % força a atualização imediata da tela (cria o efeito de animação)
        end

        pause(0.1); % pequena pausa antes de iniciar a interpolação para a próxima postura-chave
    end
end
```


## 11. Conclusão

Este ambiente permite simulação completa do Otto com:

- WSL2 + Docker Desktop como base de virtualização
- VSCode com extensões de WSL, Containers, ROS e Python para edição e depuração
- ROS 2 Jazzy
- Gazebo Harmonic
- RViz
- controle de marcha
- análise matemática em MATLAB/Octave

```note
Lembre-se:
- da separação **imagem (ambiente) vs. repositório (código)** e
- do passo de recompilação (`colcon build`) sempre que o `src/` do host for atualizado.
```
